In [ ]:
# nbplay runs entirely in your browser: install the bundled wheel first.
%pip install -q nbplay


# 13 DAW Playground

One notebook cell builds a small in-Jupyter DAW: global transport, multitrack timeline, sequencers, sampler pads, keyboard input, mixer channel strips, built-in insert effects, and browser recording.


In [ ]:
import math

import ipywidgets as widgets
from IPython.display import display

from nbplay import (
    EffectPlugin,
    KeyboardWidget,
    PadWidget,
    SamplerWidget,
    SequencerWidget,
    Session,
    SettingsWidget,
    SynthWidget,
)


def decaying_tone(freq=440, seconds=0.22, sample_rate=44100, drive=0.9):
    frames = int(sample_rate * seconds)
    return [
        math.sin(2 * math.pi * freq * i / sample_rate) * drive * (1 - i / frames) ** 2
        for i in range(frames)
    ]

settings = SettingsWidget(sample_rate=44100, channels=2, buffer_size=512)
session = Session(bpm=126.0, time_signature=(4, 4))
session.timeline.length = 64
session.timeline.count_in_bars = 1
session.timeline.auto_extend_recording = True
session.timeline.recording_extend_bars = 16

lead = SynthWidget(oscillator_type="saw", frequency=440.0, amplitude=0.42)
bass = SynthWidget(oscillator_type="square", frequency=110.0, amplitude=0.46)
drums = SamplerWidget(attack=0.002, decay=0.06, sustain=0.2, release=0.08, pad_count=8, max_voices=8)
drums.load_sample(decaying_tone(96, seconds=0.28, drive=1.0), sample_rate=44100, root_note=36, name="Kick-ish")

lead_seq = SequencerWidget(length=16, num_voices=2, bpm=session.transport.bpm)
for i, note in enumerate([72, 76, 79, 84, 83, 79, 76, 72, 71, 74, 78, 83, 81, 78, 74, 71]):
    lead_seq.set_step(i, note=note, velocity=88 + (i % 4) * 6, active=i % 2 == 0, voice=0)
for i, note in enumerate([60, 60, 64, 64, 67, 67, 64, 64, 59, 59, 62, 62, 66, 66, 62, 62]):
    lead_seq.set_step(i, note=note, velocity=58, active=i % 4 == 0, voice=1)

bass_seq = SequencerWidget(length=16, bpm=session.transport.bpm)
for i, note in enumerate([36, 36, 43, 36, 34, 34, 41, 34, 31, 31, 38, 31, 34, 34, 41, 34]):
    bass_seq.set_step(i, note=note, velocity=104 if i % 4 == 0 else 72, active=i % 2 == 0)

drum_seq = SequencerWidget(length=16, bpm=session.transport.bpm)
for i in [0, 4, 8, 12]:
    drum_seq.set_step(i, note=36, velocity=118, active=True)
for i in [3, 7, 11, 15]:
    drum_seq.set_step(i, note=38, velocity=72, active=True)

# Instrument lanes default their record source to a mixer-channel tap, so
# global record bounces each instrument to an audio clip. The vocal lane
# has no instrument and records from the microphone.
lead_track = session.add_track("Lead", lead_seq, lead, armed=True)
bass_track = session.add_track("Bass", bass_seq, bass, armed=True)
drum_track = session.add_track("Drums", drum_seq, drums, armed=True)
vocal_track = session.add_track("Vocals", armed=True)

session.mixer.set_channel_gain(lead_track.mixer_channel, 0.74)
session.mixer.set_channel_pan(lead_track.mixer_channel, 0.18)
session.mixer.add_channel_effect(lead_track.mixer_channel, EffectPlugin("filter", filter_type="lowpass", frequency=3600, q=0.7))
session.mixer.add_channel_effect(lead_track.mixer_channel, EffectPlugin("reverb", seconds=1.4, decay=2.6, wet=0.16))

session.mixer.set_channel_gain(bass_track.mixer_channel, 0.88)
session.mixer.set_channel_pan(bass_track.mixer_channel, -0.12)
session.mixer.add_channel_effect(bass_track.mixer_channel, EffectPlugin("compressor", threshold=-24, ratio=5, release=0.2))

session.mixer.set_channel_gain(drum_track.mixer_channel, 0.68)
session.mixer.add_channel_effect(drum_track.mixer_channel, EffectPlugin("compressor", threshold=-18, ratio=6, attack=0.003, release=0.12))
session.mixer.add_channel_effect(drum_track.mixer_channel, EffectPlugin("delay", time=0.125, feedback=0.18, wet=0.12))
session.mixer.master_gain = 0.82
session.mixer.add_master_effect(EffectPlugin("limiter", threshold=-1, release=0.04))

timeline_tracks = [dict(track) for track in session.timeline.tracks]
timeline_tracks[vocal_track.mixer_channel]["monitor"] = True
session.timeline.tracks = timeline_tracks

# Loop the first four bars while tracking; the ruler shows the brace and
# every sequencer follows the same loop through the shared session clock.
session.transport.loop_start_bar = 0
session.transport.loop_end_bar = 4
session.transport.loop_enabled = True

keyboard = KeyboardWidget(upper_octave=4, lower_octave=3, velocity=96)
keyboard.connect_sequencer(lead_seq)
keyboard.connect_sampler(drums)

pads = PadWidget(rows=2, cols=4, velocity=116)
pads.connect_sampler(drums)

transport_tab = widgets.VBox([settings, session.transport, session.timeline])
source_tab = widgets.VBox([lead, bass, drums])
sequence_tab = widgets.VBox([lead_seq, bass_seq, drum_seq])
input_tab = widgets.VBox([keyboard, pads])

tabs = widgets.Tab([transport_tab, source_tab, sequence_tab, input_tab, session.mixer])
for index, title in enumerate(["Transport", "Sources", "Sequencers", "Inputs", "Mixer"]):
    tabs.set_title(index, title)

display(tabs)
print("Armed lanes:", [(t["name"], t["input"]) for t in session.timeline.tracks if t["armed"]])
print(session)


Press **Rec** on the transport (or the timeline). Every armed lane records at once against the shared clock: the Lead, Bass, and Drums lanes bounce their mixer channels to audio clips, and the Vocals lane captures the microphone with a one-bar count-in. Press **Stop Rec** to keep the takes; they play back through the session mixer while the notebook view is alive.

On the timeline, drag the loop brace ends to change the loop range (the transport follows), set **Zoom** to widen the lanes and scroll, drag clips to move them between lanes, trim their edges, and use **Dup** / **Delete** on the selected clip.

To keep a take beyond the browser session, pull its bytes into Python and write them to disk; to bring audio back in, push bytes (or a file) into a lane:

```python
take = session.timeline.clips[-1]
session.timeline.export_clip(take["id"])           # bytes arrive in exported_clip_data
session.timeline.write_exported_clip("take.webm")  # once they have arrived
session.timeline.import_clip("take.webm", name="Take (file)", track_index=vocal_track.mixer_channel, start=16)
```


## Drum Zones

The drum sampler keeps its main sample for every note, and zones put a dedicated hit on single pads. Each zone has its own audio, root note, and key range; a matching zone plays instead of the main sample. In the sampler view, the **Rec** button next to the zones records the microphone straight onto the selected pad.

In [ ]:
def noise_burst(seconds=0.16, sample_rate=44100, seed=7):
    frames = int(sample_rate * seconds)
    state = seed
    out = []
    for i in range(frames):
        state = (state * 1103515245 + 12345) % 2**31
        out.append((state / 2**30 - 1.0) * (1 - i / frames) ** 3)
    return out

drums.add_zone(decaying_tone(60, seconds=0.32, drive=1.0), 36, 36, name="Kick")
drums.add_zone(noise_burst(), 38, 38, name="Snare")
drums.add_zone(noise_burst(seconds=0.05, seed=3), 42, 42, name="Hat")
print([(zone["name"], zone["note_low"]) for zone in drums.zones])

## Song Mode

Register the drum pattern and a fill as named patterns, then chain them onto the Drums lane. The clips play through the drum sampler and the drum sequencer stops looping on its own, so the arrangement drives that track. Four patterns of one bar each fit the four-bar loop above; `update_pattern()` regenerates every clip made from a name after you edit it.

In [ ]:
fill_seq = SequencerWidget(length=16, bpm=session.transport.bpm)
for i in range(0, 16, 2):
    fill_seq.set_step(i, note=36 if i % 4 == 0 else 38, velocity=90 + i * 2, active=True)

session.add_pattern("groove", drum_seq)
session.add_pattern("fill", fill_seq)
song = session.chain(drum_track, ["groove", "groove", "groove", "fill"])
print([(clip["name"], clip["start"]) for clip in song])
print("drum sequencer follows transport:", drum_seq.follow_transport)

## Take to Pad

After recording, a take on the timeline can become a drum pad. `capture_clip()` hands the clip to the sampler in the browser, which decodes it with the clip's trim and adds a zone on that pad's note. Rerun this cell once a take exists.

In [ ]:
takes = [clip for clip in session.timeline.clips if clip["kind"] == "audio" and clip.get("audio_url")]
if takes:
    drums.capture_clip(session.timeline, takes[-1]["id"], pad=3)
    print("Capturing", takes[-1]["name"], "onto pad 4")
else:
    print("Record a take first, then rerun this cell")

## MIDI In and Out

A hardware controller plugs in through `MidiKeyboardWidget`; its keys play the drum sampler and `MidiLearn` maps its knobs to widget traits (the mod wheel to the lead level, controller 74 to the tempo). `MidiOutputWidget` sends everything the session plays, sequencers and clips included, to a Web MIDI output port at the scheduled time. Pick the ports in the widgets below; both need a browser with Web MIDI.

In [ ]:
from nbplay import MidiKeyboardWidget, MidiLearn, MidiOutputWidget

midi_keys = MidiKeyboardWidget(session_id=session.session_id, channel_index=drum_track.mixer_channel)
midi_keys.connect_sampler(drums)
learn = MidiLearn(midi_keys)
learn.map(1, lead, "amplitude")
learn.map(74, session.transport, "bpm", low=80.0, high=160.0)

midi_out = MidiOutputWidget(session_id=session.session_id)
display(widgets.VBox([midi_keys, midi_out]))